# Data quality — the pipeline's first step

- **Part 1 runs before notebook 1**, on the raw questionnaires, and stops the
  pipeline there.
- It reads every raw questionnaire — the same files notebook 1 reads — and
  finds what is worth a human's judgement before a full run:
  - **labels the dictionary has never seen** — a column header or a value with
    no good match at any confidence, in either language;
  - **Value cells that are not a number and that `clean_one_value()` could not
    make sense of** — a reading already confirmed in `value corrections.xlsx`
    and a recognized unit phrase are resolved automatically in the real run and
    need no review; this is only the residue;
  - **structural problems** — reporting only: a duplicate column header, a
    stray space in a merge key, or a sheet with no `index` row breaks parsing
    itself and has to be fixed in the source Excel file by hand.
- **Nothing is changed here.** Running this notebook writes one file per
  chapter, **`Data quality issues before pipeline execution_<Chapter>.txt`** —
  every outstanding issue, read-only.

The loop:

1. Run this notebook. It writes one file per chapter, beside the codes folder.
2. Open the chapter(s) you are about to run and read what it found.
3. Correct whatever it calls for directly — the source questionnaire or
   `value corrections.xlsx` (in `DATA COLLECTOR\`), or
   `translation dict_V2.xlsx` (in the `COMPENDIUM ARAB SOCIETY - V2\` root) —
   there is no intermediate file to edit and apply.
4. Tell Claude to resume. Notebook 1 runs next.

In [ ]:
"""
CELL: Imports and logging setup.
"""
import difflib
import logging
import re
from collections import defaultdict
from datetime import date
from pathlib import Path

import pandas as pd
import openpyxl
from openpyxl.utils import get_column_letter

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium")


## Config

In [ ]:
"""
CELL: Configuration - the same paths and constants notebook 1 uses, since this
notebook has to see exactly what notebook 1 would see.
"""
DATA_COLLECTOR_PATH = Path.home() / "OneDrive - United Nations" / "Desktop" / "DSS" / "DATA COLLECTOR"
# The V2 dictionary, in the V2 root beside the outputs - NOT the shared one
# under DATA COLLECTOR, which belongs to the original pipeline.
TRANSLATION_DICT_PATH = (Path.home() / "OneDrive - United Nations" / "Desktop" / "DSS"
                         / "COMPENDIUM ARAB SOCIETY - V2" / "translation dict_V2.xlsx")
VALUE_CORRECTIONS_PATH = DATA_COLLECTOR_PATH / "value corrections.xlsx"
COMPENDIUM_PATH = Path.home() / "OneDrive - United Nations" / "Desktop" / "DSS" / "COMPENDIUM ARAB SOCIETY - V2"

QUESTIONNAIRE_PREFIX = "datacollector_received_quest_"
LANGUAGES = ["AR", "EN"]
DEFAULT_LANGUAGE = "AR"

# Leave as None to check every chapter found, or restrict e.g. ["Population"].
CHAPTERS = None

# A fuzzy match must score at least this well (0 to 1) to be used automatically
# by the real pipeline. Below it is exactly what this notebook reviews - same
# number as notebook 1's, because a label just above the line needs no review
# and one just below it does.
FUZZY_MATCH_CUTOFF = 0.6

# Column names the pipeline creates itself, written here in Arabic.
YEAR_COLUMN = "السنة"
VALUE_COLUMN = "العدد"
CHAPTER_COLUMN = "الفصل"

MERGE_COLUMNS = ["السنة", "المؤشر", "الدولة"]

# Never fuzzy-matched by the real pipeline either: two citations differing by
# one digit score high enough to overwrite each other. A Source value still
# gets reviewed here if the dictionary has no *exact* translation for it, but
# never with a fuzzy-matched suggestion pre-filled.
COLUMNS_NOT_FUZZY_MATCHED = ["المصدر"]



## The dictionary

In [ ]:
"""
CELL: Load translation dict.xlsx - identical to notebook 1's own loader, since
a label reviewed here has to be checked against the exact same vocabulary
notebook 1 will use.
"""


def load_dictionary():
    dict_df = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")

    column_map, value_map = {}, {}
    for arabic_column in dict_df["col_ar"].dropna().unique():
        rows = dict_df[dict_df["col_ar"] == arabic_column]
        column_map[arabic_column] = rows["col_en"].iloc[0]
        value_map[arabic_column] = {
            arabic: english
            for arabic, english in zip(rows["val_ar"], rows["val_en"])
            if pd.notna(arabic)
        }

    english_columns = {}
    english_values = {}
    for english_column in dict_df["col_en"].dropna().unique():
        rows = dict_df[dict_df["col_en"] == english_column]
        english_columns[english_column] = english_column
        english_values[english_column] = {
            str(v): str(v) for v in rows["val_en"].dropna().unique()
        }

    chapter_rows = dict_df[dict_df["col_en"] == "Chapter"]
    chapter_to_arabic = dict(zip(chapter_rows["val_en"], chapter_rows["val_ar"]))

    return (column_map, value_map), (english_columns, english_values), chapter_to_arabic


DICTIONARY_AR_TO_EN, ENGLISH_VOCABULARY, CHAPTER_TO_ARABIC = load_dictionary()


def vocabulary(language):
    """The known column names and values for a language, as
    (column_names, values_by_column) - what a raw label is checked against."""
    return DICTIONARY_AR_TO_EN if language == "AR" else ENGLISH_VOCABULARY


def column_name_for(english_name, language):
    """Any dictionary column, spelled for the given language."""
    if language == "EN":
        return english_name
    english_to_arabic = {en: ar for ar, en in DICTIONARY_AR_TO_EN[0].items()}
    return english_to_arabic.get(english_name, english_name)


def column_name(arabic_name, language):
    """One of the pipeline's own column names, spelled for the given language."""
    arabic_to_english, _ = DICTIONARY_AR_TO_EN
    return arabic_name if language == "AR" else arabic_to_english[arabic_name]


def known_english_columns():
    """Every English column name the dictionary knows - the vocabulary a
    column-name finding's correction has to be one of."""
    arabic_to_english, _ = DICTIONARY_AR_TO_EN
    return sorted(set(arabic_to_english.values()))


arabic_columns, _ = DICTIONARY_AR_TO_EN
logger.info(f"Dictionary loaded: {len(arabic_columns)} column names, Arabic -> English")


## Recognizing a Value cell notebook 1 could not make sense of

In [ ]:
"""
CELL: clean_one_value() - identical to notebook 1's, so "not a number" means
the same thing here as it will on the real run.

Only clean_one_value() itself is needed, not the table-level clean_values():
this notebook checks one raw cell at a time as it reads each sheet, rather
than a whole reshaped column at once.
"""

NUMBER_IN_TEXT = re.compile(r"[-+]?\d[\d\s, ]*(?:\.\d+)?")

UNIT_MULTIPLIERS = [
    (re.compile(r"بالمليون|بالملايين"), 1_000_000, "in millions"),
    (re.compile(r"بالال?ف"), 1_000, "in thousands"),
]


def load_value_corrections():
    """Raw Value text a person has already confirmed a reading for, from a
    previous run of this same notebook's review-and-apply loop. A cell that
    matches one is not a new finding - it is already answered, and
    clean_one_value() below returns that answer instead of flagging it again.
    """
    if not VALUE_CORRECTIONS_PATH.exists():
        return {}
    table = pd.read_excel(VALUE_CORRECTIONS_PATH, engine="openpyxl")
    corrections = {}
    for chapter, raw, corrected in zip(
        table["chapter"], table["raw_value"], table["corrected_value"]
    ):
        if pd.isna(chapter) or pd.isna(raw) or pd.isna(corrected):
            continue
        corrections[(str(chapter).strip(), str(raw).strip())] = str(corrected).strip()
    return corrections


VALUE_CORRECTIONS = load_value_corrections()
if VALUE_CORRECTIONS:
    logger.info(f"Value corrections already on file: {len(VALUE_CORRECTIONS):,} "
                f"(those cells are already answered and will not be reviewed again)")


def unit_multiplier(text):
    for pattern, factor, meaning in UNIT_MULTIPLIERS:
        if pattern.search(text):
            return factor, meaning
    return 1, None


def scaled_text(number, factor):
    value = float(number) * factor
    return str(int(value)) if value.is_integer() else str(round(value, 10))


def clean_one_value(raw, chapter=None):
    """Return (cleaned, note, factor) - see notebook 1's cell of the same name.

    Only two corrections are left there, now that the questionnaire workbooks
    carry validation rules of their own: a reading already confirmed in
    value corrections.xlsx, and a recognized unit phrase. Everything else that
    is not a number is left exactly as it is and reported - which is what this
    notebook then puts in front of a person.

    This copy exists so this notebook needs nothing from notebook 1 at import
    time; keep the two in step by hand if either changes.
    """
    if pd.isna(raw):
        return raw, None, 1

    text = str(raw).replace("\xa0", " ").strip()
    if text == "":
        if str(raw) == "":
            return raw, None, 1
        return raw, f"{str(raw)!r} holds only whitespace and was left as it is", 1

    plain = re.sub(r"[\s,]", "", text)
    try:
        float(plain)
        return raw, None, 1
    except ValueError:
        pass

    confirmed = VALUE_CORRECTIONS.get((chapter, text))
    if confirmed is not None:
        return confirmed, f"{text!r} -> {confirmed} (confirmed reading, from value corrections.xlsx)", 1

    match = NUMBER_IN_TEXT.search(text)
    if match:
        number = re.sub(r"[\s, ]", "", match.group(0))
        removed = text.replace(match.group(0), "").strip()
        factor, meaning = unit_multiplier(removed) if removed else (1, None)
        if factor != 1:
            try:
                value = scaled_text(number, factor)
            except ValueError:
                return raw, f"{text!r} is not a number and was left as it is", 1
            return value, (f"{text!r} -> {value} (dropped {removed!r}, meaning {meaning} "
                           f"- multiplied by {factor:,})"), factor

    return raw, f"{text!r} is not a number and was left as it is", 1


def needs_review(raw, chapter=None):
    """True only for the residue clean_one_value() could not make sense of -
    everything else is either already fine or already handled automatically
    by the real pipeline, and reviewing it here would just be noise.
    """
    cleaned, note, _ = clean_one_value(raw, chapter)
    return note is not None and str(cleaned) == str(raw)

## The three checks

In [ ]:
"""
CELL: check_labels(), check_values(), check_structure() - the three checks,
run once per sheet as it is read. Nothing here corrects anything; each only
records what it finds.
"""

# Raw hits, one per unique value per sheet - the same granularity
# correct_with_dictionary() itself works at. Aggregated across the whole run
# into one finding per (chapter, column, raw text) before the review file is
# written, so a label repeated across fifty sheets appears once.
LABEL_HITS = []
VALUE_HITS = []
STRUCTURE_PROBLEMS = []


def best_match(text, choices):
    best_choice, best_score = None, -1
    for choice in choices:
        score = difflib.SequenceMatcher(None, str(text), str(choice)).ratio()
        if score > best_score:
            best_choice, best_score = choice, score
    return best_choice, best_score


def check_labels(table, language, chapter, file_name, sheet_name):
    """Column names and values with no good match in the dictionary - the
    same two passes as notebook 1's correct_with_dictionary(), except nothing
    is renamed and only the below-cutoff cases are kept. A label notebook 1
    would fix on its own needs no review; this is only what it would give up
    on and leave as it found it.
    """
    known_columns, known_values_by_column = vocabulary(language)
    never_guessed = [column_name(c, language) for c in COLUMNS_NOT_FUZZY_MATCHED]

    for column in table.columns:
        if column in known_columns:
            continue

        match, score = best_match(column, known_columns.keys())
        if match is None or score < FUZZY_MATCH_CUTOFF:
            LABEL_HITS.append({
                "scope": "column", "chapter": chapter, "col_ar": None,
                "raw": column, "language": language,
                "closest": match, "closest_en": None, "score": score,
                "file": file_name, "sheet": sheet_name,
            })

    for column in table.columns:
        if column in never_guessed:
            # Source is only ever a translation concern on the ARABIC side -
            # an English questionnaire's citation is already the target
            # language and needs nothing. Even there, plenty of Arabic
            # questionnaires cite an English source ("MICS 2022", a URL) -
            # already correct as it stands, exactly like notebook 4's own
            # find_untranslated() treats it. What is left after both of those
            # is flagged, but never with a fuzzy-matched suggestion: two
            # citations differing by one digit would otherwise score high
            # enough to suggest one for the other.
            if language == "AR":
                known_values = known_values_by_column.get(column) or {}
                for value in table[column].dropna().unique():
                    if value in known_values or not looks_arabic(value):
                        continue
                    LABEL_HITS.append({
                        "scope": "source", "chapter": chapter, "col_ar": column,
                        "raw": value, "language": language,
                        "closest": None, "closest_en": None, "score": None,
                        "file": file_name, "sheet": sheet_name,
                    })
            continue

        known_values = known_values_by_column.get(column)
        if not known_values:
            continue  # not a dictionary column, or has no fixed vocabulary (e.g. Year, Value)

        for value in table[column].dropna().unique():
            if value in known_values:
                continue

            match, score = best_match(value, known_values.keys())
            if match is None or score < FUZZY_MATCH_CUTOFF:
                LABEL_HITS.append({
                    "scope": "value", "chapter": chapter, "col_ar": column,
                    "raw": value, "language": language,
                    "closest": match, "closest_en": known_values.get(match) if match else None,
                    "score": score, "file": file_name, "sheet": sheet_name,
                })


def check_values(table, language, chapter, file_name, sheet_name):
    """Value cells that are not a number and that clean_one_value() could not
    make sense of on its own - see needs_review() above for exactly which
    cases that is.
    """
    value_column = column_name(VALUE_COLUMN, language)
    if value_column not in table.columns:
        return

    indicator_column = column_name_for("Indicator", language)
    country_column = column_name_for("Country", language)
    year_column = column_name(YEAR_COLUMN, language)

    for position, raw in table[value_column].items():
        if pd.isna(raw) or not needs_review(raw, chapter):
            continue
        where = {}
        for field, column in [("country", country_column),
                              ("indicator", indicator_column),
                              ("year", year_column)]:
            if column in table.columns:
                where[field] = table.at[position, column]
        VALUE_HITS.append({
            "chapter": chapter, "raw": str(raw).replace("\xa0", " ").strip(),
            "file": file_name, "sheet": sheet_name, **where,
        })


# Cover / metadata tabs have no "index" header row and are skipped by design
# - not a problem, just not a data sheet. Matched by name because there is
# nothing else to go on: a tab with no header row looks structurally
# identical whether it is an expected cover page or a genuinely broken data
# sheet, and only the name tells the two apart (Iraq's and Jordan's legacy
# Health sheets have no "index" row either, and ARE a real problem - see
# Known issues in CLAUDE.md). The metadata tab's name varies by file
# (a leading "_", a trailing " SDG", "_sdg" vs "_SDG", "المفصلة" vs
# "المجمعة") but always contains this core phrase, so it is matched as a
# substring rather than a fixed set of exact names.
COVER_TAB_NAMES = {"العنوان", "قائمة الجداول", "كيفية الإستخدام"}
COVER_TAB_MARKER = "البيانات الوصفية"


def is_cover_tab(sheet_name):
    name = str(sheet_name).strip()
    return name in COVER_TAB_NAMES or COVER_TAB_MARKER in name


def check_structure_error(chapter, language, file_name, sheet_name, error):
    """A sheet (or file) that could not be read at all."""
    STRUCTURE_PROBLEMS.append({
        "chapter": chapter, "language": language, "file": file_name,
        "sheet": sheet_name, "problem": "cannot read the sheet",
        "detail": f"{type(error).__name__}: {error}",
    })


def check_structure_raw(raw_sheet, header_rows, chapter, language, file_name, sheet_name):
    """Problems that do not raise an exception but corrupt the sheet anyway -
    ported from the old Compendium_Data_Quality.ipynb's
    check_questionnaires(). A duplicate column header does not crash
    extract_tables(); it silently produces a stray extra column instead, which
    is exactly why this cannot wait to be caught as an error. Reporting only:
    every one of these has to be fixed in the source Excel file, not here.
    """
    merge_keys = {"السنة", "المؤشر", "الدولة", "Year", "Indicator", "Country"}
    here = {"chapter": chapter, "language": language, "file": file_name, "sheet": sheet_name}

    names = raw_sheet.iloc[header_rows[0]].dropna().astype(str).str.strip().tolist()

    duplicated = {n for n in names if names.count(n) > 1}
    if duplicated:
        STRUCTURE_PROBLEMS.append({**here, "problem": "duplicate column header",
                                   "detail": f"{sorted(duplicated)} - the second becomes a stray column"})

    year_like = [n for n in names if re.fullmatch(r"\d{4}(\.\d+)?", n) and not n.isdigit()]
    if year_like:
        STRUCTURE_PROBLEMS.append({**here, "problem": "year column is not a plain number",
                                   "detail": f"{year_like} - will be treated as a label, not unpivoted"})

    if not any(n.isdigit() for n in names):
        STRUCTURE_PROBLEMS.append({**here, "problem": "no year columns", "detail": "nothing to unpivot"})

    positions = raw_sheet.iloc[header_rows[0]]
    for position, name in positions.items():
        if not isinstance(name, str) or name.strip() not in merge_keys:
            continue
        if name != name.strip():
            STRUCTURE_PROBLEMS.append({**here, "problem": "whitespace in a merge-key header",
                                       "detail": f"{name!r} - breaks the source merge"})
        body = raw_sheet.index[raw_sheet[0] == "1"]
        values = raw_sheet.loc[body, position].dropna()
        untidy = [v for v in values.unique() if isinstance(v, str) and v != v.strip()]
        if untidy:
            STRUCTURE_PROBLEMS.append({**here, "problem": "whitespace in merge-key values",
                                       "detail": f"{len(untidy)} value(s), e.g. {untidy[0]!r}"})


## Reading a raw sheet

In [ ]:
"""
CELL: Reading a raw sheet - identical to notebook 1's extract/detect/reshape
cells, so this notebook checks exactly the table notebook 1 would build,
before any correction is applied to it. No file is written from here.
"""


def questionnaire_folders():
    found = []
    for folder in sorted(DATA_COLLECTOR_PATH.glob(f"{QUESTIONNAIRE_PREFIX}*")):
        if not folder.is_dir():
            continue
        language = folder.name[len(QUESTIONNAIRE_PREFIX):].strip().upper()
        if language not in LANGUAGES:
            logger.warning(f"Skipping {folder.name}: '{language}' is not one of {LANGUAGES}")
            continue
        found.append((language, folder))
    return found


def discover_chapters():
    names = set()
    for _, questionnaire_root in questionnaire_folders():
        for child in questionnaire_root.iterdir():
            if child.is_dir() and any(
                f for f in child.glob("*.xlsx") if not f.name.startswith("~$")
            ):
                names.add(child.name)
    return sorted(names)


def chapters_to_process():
    if CHAPTERS:
        return list(CHAPTERS)
    found = discover_chapters()
    logger.info(f"Chapters discovered on disk: {found}")
    return found


def looks_arabic(text):
    """True if the text contains at least one Arabic letter. U+0600-U+06FF is
    the Arabic Unicode block; English text has nothing in it. Identical to
    notebook 1's own function of the same name."""
    return any("؀" <= character <= "ۿ" for character in str(text))


def detect_language(table):
    names = [str(c) for c in table.columns if not str(c).isdigit()]
    if not names:
        return DEFAULT_LANGUAGE
    arabic_names = sum(1 for name in names if looks_arabic(name))
    return "AR" if arabic_names > len(names) / 2 else "EN"


def extract_tables(raw_sheet):
    header_rows = raw_sheet.index[raw_sheet[0] == "index"].tolist()
    data_header_row, source_header_row = header_rows[0], header_rows[1]

    data_columns = raw_sheet.iloc[data_header_row].dropna().str.strip()
    data_table = raw_sheet[raw_sheet[0] == "1"][data_columns.index].copy()
    data_table.columns = data_columns.values
    data_table = data_table.drop(columns=["index"])

    source_columns = raw_sheet.iloc[source_header_row].dropna().str.strip()
    source_table = raw_sheet[raw_sheet[0] == "2"][source_columns.index].copy()
    source_table.columns = source_columns.values
    source_table = source_table.drop(columns=["index"])

    return data_table, source_table


def reshape_and_merge(data_table, source_table, language):
    id_columns = [c for c in data_table.columns if not str(c).isdigit()]
    year_columns = [c for c in data_table.columns if str(c).isdigit()]

    long_table = data_table.melt(
        id_vars=id_columns,
        value_vars=year_columns,
        var_name=column_name(YEAR_COLUMN, language),
        value_name=column_name(VALUE_COLUMN, language),
    )

    merge_columns = [column_name(c, language) for c in MERGE_COLUMNS]
    merge_columns = [c for c in merge_columns if c in long_table.columns and c in source_table.columns]

    long_table = long_table.copy()
    source_table = source_table.copy()
    for column in merge_columns:
        long_table[column] = long_table[column].astype(str).str.replace("\xa0", " ").str.strip()
        source_table[column] = source_table[column].astype(str).str.replace("\xa0", " ").str.strip()

    return pd.merge(long_table, source_table, on=merge_columns, how="left")


def raw_sheets(chapter, language, questionnaire_root):
    """Every (file_name, sheet_name, table) notebook 1 would build for one
    chapter folder, in the order it would build them.

    A sheet that cannot be read at all, or that reads but is structurally
    broken (duplicate header, stray whitespace, ...), is reported through
    check_structure_error() / check_structure_raw() and skipped - nothing is
    yielded for it, so callers only ever see a usable table.
    """
    folder = questionnaire_root / chapter
    if not folder.exists():
        return

    for file_path in sorted(f for f in folder.glob("*.xlsx") if not f.name.startswith("~$")):
        try:
            xls = pd.ExcelFile(file_path, engine="openpyxl")
        except Exception as error:
            check_structure_error(chapter, language, file_path.name, "-", error)
            continue

        for sheet_name in xls.sheet_names:
            try:
                raw_sheet = pd.read_excel(xls, sheet_name=sheet_name, header=None, dtype=str)
            except Exception as error:
                check_structure_error(chapter, language, file_path.name, sheet_name, error)
                continue

            header_rows = raw_sheet.index[raw_sheet[0] == "index"].tolist()
            if len(header_rows) < 2:
                if not is_cover_tab(sheet_name):
                    # No "index" row and not a recognized cover-tab name: this
                    # is what notebook 1 itself would hit as an IndexError at
                    # its "extract" step - a real problem, not an expected skip.
                    STRUCTURE_PROBLEMS.append({
                        "chapter": chapter, "language": language,
                        "file": file_path.name, "sheet": sheet_name,
                        "problem": "no index header row, and not a recognized cover tab",
                        "detail": f"only {len(header_rows)} 'index' row(s) found "
                                  f"(need 2) - likely a legacy sheet layout, needs "
                                  f"a look in the source file",
                    })
                continue

            check_structure_raw(raw_sheet, header_rows, chapter, language,
                                file_path.name, sheet_name)
            try:
                data_table, source_table = extract_tables(raw_sheet)
                table = reshape_and_merge(data_table, source_table, language)
            except Exception as error:
                check_structure_error(chapter, language, file_path.name, sheet_name, error)
                continue

            yield file_path.name, sheet_name, table


## run_checks()

In [ ]:
"""
CELL: run_checks() - read every questionnaire and collect every finding.
"""


def run_checks():
    LABEL_HITS.clear()
    VALUE_HITS.clear()
    STRUCTURE_PROBLEMS.clear()

    folders = questionnaire_folders()
    chapters = chapters_to_process()
    print(f"Questionnaire folders found: {[f'{lang} ({f.name})' for lang, f in folders]}")

    sheets_read = 0
    total_steps = len(folders) * len(chapters)
    step_number = 0
    for language, questionnaire_root in folders:
        print(f"\n=== {language}  ({questionnaire_root.name}) ===")
        for chapter in chapters:
            step_number += 1
            bar = "#" * step_number + "-" * (total_steps - step_number)
            print(f"[{bar}] {step_number}/{total_steps}  {language}/{chapter}")
            for file_name, sheet_name, table in raw_sheets(chapter, language, questionnaire_root):
                check_labels(table, language, chapter, file_name, sheet_name)
                check_values(table, language, chapter, file_name, sheet_name)
                sheets_read += 1

    logger.info(f"Read {sheets_read:,} data sheet(s)")

    return sheets_read


## write_chapter_reports() - one brief, read-only file per chapter


In [ ]:
"""
CELL: format_locations() and write_chapter_reports() - Part 1's one output, a
short read-only file per chapter.

Nothing here is ever applied automatically, and nothing here is aggregated
across chapters (the same mistyped column header in both Housing and Poverty
gets its own line in each chapter's own file, not one merged entry - a
per-chapter file that silently dropped the other chapter's copy would be
misleading). It exists to answer one question per chapter: what got flagged,
on which row, and why did it need a person instead of the real pipeline's own
matching? A person reads it, corrects whatever it calls for directly (the
source questionnaire, translation dict.xlsx, or value corrections.xlsx), and
tells Claude to resume.

Label and structural findings are about a column, a source citation, or a
whole sheet - not one data row - so Country/Indicator/Year are printed as "-"
for those; only a value finding has a row to point at.
"""


def format_locations(locations, limit=3):
    """A short, readable sample of where a finding was seen.

    Each location is (file, sheet) for a value finding, or (chapter, file,
    sheet) for a label finding - only the last two elements are shown, since
    a label's chapter is already named on its own line.
    """
    shown = list(dict.fromkeys(locations))  # de-duplicate, keep order
    text = ", ".join(f"{loc[-2]} ({loc[-1]})" for loc in shown[:limit])
    if len(shown) > limit:
        text += f", and {len(shown) - limit} more"
    return text


CHAPTER_REPORT_PREFIX = "Data quality issues before pipeline execution_"


def issue_reason(finding):
    """Why this finding needed a person, not the real pipeline's own matching."""
    if "problem" in finding:
        return f"{finding['problem']} - {finding['detail']}"
    if finding.get("scope") == "source":
        return ("no exact translation, and Source is never fuzzy-matched - two "
                "citations differing by one digit would otherwise be treated as "
                "the same")
    if "scope" in finding:
        if finding["closest"] is None:
            return "no dictionary entry close enough for difflib to suggest one"
        return (f"closest dictionary entry is {finding['closest']!r} "
                f"(score {finding['score']:.2f}), below the "
                f"{FUZZY_MATCH_CUTOFF:.2f} cutoff the real pipeline fixes above")
    return "not a number, and clean_one_value() could not make sense of it alone"


def chapter_label_findings(chapter):
    """LABEL_HITS for one chapter, deduplicated within that chapter only -
    unlike aggregate_labels(), which merges the same label across chapters."""
    seen = {}
    for hit in LABEL_HITS:
        if hit["chapter"] != chapter:
            continue
        key = (hit["scope"], hit["col_ar"], hit["raw"])
        seen.setdefault(key, hit)
    return list(seen.values())


def chapter_value_findings(chapter):
    """VALUE_HITS for one chapter, deduplicated by raw text, keeping the first
    occurrence's row (country/indicator/year) as the example."""
    seen = {}
    order = []
    for hit in VALUE_HITS:
        if hit["chapter"] != chapter:
            continue
        if hit["raw"] not in seen:
            seen[hit["raw"]] = {**hit, "locations": []}
            order.append(hit["raw"])
        seen[hit["raw"]]["locations"].append((hit["file"], hit["sheet"]))
    return [seen[key] for key in order]


def write_chapter_reports(chapters, folder=None):
    folder = folder or COMPENDIUM_PATH
    stamp = pd.Timestamp.now().strftime("%d %B %Y, %H:%M")
    written = []

    for chapter in chapters:
        labels = chapter_label_findings(chapter)
        values = chapter_value_findings(chapter)
        structure = [row for row in STRUCTURE_PROBLEMS if row["chapter"] == chapter]

        lines = [
            f"DATA QUALITY ISSUES BEFORE PIPELINE EXECUTION - {chapter}",
            "=" * 78,
            f"generated {stamp}, from the raw questionnaires. Read-only summary -",
            "every outstanding issue lives here; correct it directly (the source",
            "questionnaire, translation dict.xlsx, or value corrections.xlsx) and",
            "tell Claude to resume. Nothing here is ever applied automatically.",
            "",
            f"{len(labels)} label issue(s), {len(values)} value issue(s), "
            f"{len(structure)} structural issue(s)",
        ]

        if not (labels or values or structure):
            lines += ["", "Nothing found."]

        if labels:
            lines += ["", "-" * 78, "LABELS NOT IN THE DICTIONARY, EXACTLY", "-" * 78]
            for i, hit in enumerate(labels, start=1):
                lines.append(
                    f"{i}. Country: -  |  Sheet: {hit['file']} ({hit['sheet']})  |  "
                    f"Indicator: -  |  Year: -"
                )
                lines.append(f"   {hit['raw']!r} - {issue_reason(hit)}")

        if values:
            lines += ["", "-" * 78, "VALUES clean_one_value() COULD NOT PARSE", "-" * 78]
            for i, finding in enumerate(values, start=1):
                where = format_locations(finding["locations"])
                lines.append(
                    f"{i}. Country: {finding.get('country', '-')}  |  Sheet: {where}  |  "
                    f"Indicator: {finding.get('indicator', '-')}  |  "
                    f"Year: {finding.get('year', '-')}"
                )
                lines.append(f"   {finding['raw']!r} - {issue_reason(finding)}")

        if structure:
            lines += ["", "-" * 78, "STRUCTURAL PROBLEMS (fix in the source Excel file)", "-" * 78]
            for i, row in enumerate(structure, start=1):
                lines.append(
                    f"{i}. Country: -  |  Sheet: {row['file']} ({row['sheet']})  |  "
                    f"Indicator: -  |  Year: -"
                )
                lines.append(f"   {issue_reason(row)}")

        lines.append("")
        path = folder / f"{CHAPTER_REPORT_PREFIX}{chapter}.txt"
        path.write_text("\n".join(lines), encoding="utf-8")
        written.append(path)

    return written


## Run - Part 1

In [ ]:
"""
CELL: Main run - read every questionnaire and external file, write each
chapter's own issue report, and stop.
"""
run_checks()

chapter_report_paths = write_chapter_reports(chapters_to_process())
print("\n" + "=" * 70)
print("PER-CHAPTER ISSUE REPORTS WRITTEN")
print("=" * 70)
for p in chapter_report_paths:
    print(f"  {p}")

print("\n" + "#" * 70)
print("# STOP - this is the pipeline's first step, and it gates the rest.")
print("# Review each file above, correct directly whatever it calls for")
print("# (the source questionnaire, translation dict.xlsx, or value")
print("# corrections.xlsx), then tell Claude to resume before notebook 1")
print("# runs.")
print("#" * 70)


## Part 2 — the final files: completeness and contradictions

- Part 1 above reads the raw questionnaires, before any of the pipeline has
  touched them.
- Part 2 reads the opposite end — the finished `<Chapter>_EN.xlsx` files — and
  answers a different question: not "is this label right", but "is the finished
  data any good".

Two things, in one pass per chapter:

- **Completeness.**
  - For every indicator × country, how many of the expected years
    (`FIRST_YEAR` to the current edition) carry a value.
  - Banded `none` / `sparse` / `partial` / `strong` / `complete` rather than
    averaged into one number — a one-point series and a full one must never be
    allowed to cancel out into something that reads as "half full".
  - Which breakdowns a country actually supplies for an indicator, too.
- **Contradictions.**
  - A reported total that does not match the sum of its own parts.
  - A year-on-year change of more than 10×.
  - A percentage indicator that also holds absolute counts.
  - A percentage breakdown that does not add to 100.
  - A negative value, or a percentage outside 0-100.
  - The same row reported twice with two different values.

- **Nothing here suggests a correction the way Part 1 does** — a spike or a
  contradiction in the FINISHED data is a finding about the SOURCE, and fixing
  it means going back to the questionnaire, not editing anything downstream.
- `data_gaps_report.xlsx` is read-only output.

### Config

In [ ]:
"""
CELL: Configuration for the final-file checks - completeness and
contradictions. COMPENDIUM_PATH is already set, above.
"""
import json

CODES_PATH = COMPENDIUM_PATH / "codes"
PROJECT_PATH = CODES_PATH / "data quality"
GAPS_REPORT_PATH = PROJECT_PATH / "data_gaps_report.xlsx"

# Every long file lives here - one folder, both languages. Only the EN side is
# read below: every check here is written against English column names.
LONG_FILES_PATH = COMPENDIUM_PATH / "longfiles"

# The years the compendium expects a figure for - 2010 to the current edition,
# taken from the calendar rather than typed in, so this does not quietly go on
# measuring last year's range. The template's projection columns run past it and
# are not counted as gaps, because they sit beyond LAST_YEAR by construction.
# Pin LAST_YEAR to an integer to re-measure an older edition.
FIRST_YEAR = 2010
LAST_YEAR = date.today().year
EXPECTED_YEARS = list(range(FIRST_YEAR, LAST_YEAR + 1))

# Columns that are not a disaggregation.
FIXED = {"Indicator", "Country", "Chapter", "Year", "Value", "Source"}

# The value meaning "all of them", per dimension. A dimension absent from this
# map has no total by design - an indicator broken down that way (causes of
# death, economic activity, ...) has no meaningful aggregate row, so leaving
# it out is deliberate, not a gap in the map.
TOTAL_LABELS = {
    "Sex": "Both sexes",
    "Age Group": "Age Total",
    "Area": "Area Total",
    "Nationality": "Nationality Total",
    "Marital status": "Marital status Total",
    "Quintile": "Total",
    "Types of products/services": "Total",
}

# A year-on-year change of more than this is flagged. Deliberately loose: real
# populations do not change tenfold in a year, so a hit is a transcription
# error, a units change, or a break in the series worth a footnote.
SPIKE_FACTOR = 10

# How far a reported total may stray from the sum of its parts, as a fraction.
TOTAL_TOLERANCE = 0.01

# How far a set of percentages may stray from 100, in points.
PERCENT_TOLERANCE = 1.0

# Series are grouped by how many of the 17 expected years they actually carry.
# Reporting one average percentage would let a one-point series and a full one
# cancel out into a number that reads as "half full"; these bands keep the
# difference visible. Ordered worst to best.
POINT_BANDS = [
    ("none", 0, 0),          # never reported at all
    ("sparse", 1, 6),        # too few points to read as a series
    ("partial", 7, 12),
    ("strong", 13, 16),
    ("complete", 17, 17),    # every expected year present
]


def band_of(points):
    for name, low, high in POINT_BANDS:
        if low <= points <= high:
            return name
    return "none"


### Helpers

In [ ]:
"""
CELL: Helpers for the final-file checks.

to_number() here is a lighter check than clean_one_value() above: this side
only ever needs a float or None for measurement across tens of thousands of
already-processed rows, never a suggested correction, so it does not carry
clean_one_value()'s unit-phrase, placeholder or sum-expression logic.
"""


def to_number(value):
    if pd.isna(value):
        return None
    text = str(value).replace("\xa0", " ").replace(",", "").strip()
    text = re.sub(r"\s+", "", text)
    if text in ("", "-", "--", "..", "..."):
        return None
    try:
        return float(text)
    except ValueError:
        return None


def headline_mask(table, dimensions):
    """True where a row is the headline figure: every dimension either empty or
    sitting at its own total. A dimension with no total label must be empty."""
    mask = pd.Series(True, index=table.index)
    for dimension in dimensions:
        column = table[dimension]
        total = TOTAL_LABELS.get(dimension)
        ok = column.isna()
        if total is not None:
            ok = ok | (column.astype(str).str.strip() == total)
        mask &= ok
    return mask


def missing_years_and_longest_gap(present):
    """Which expected years are absent, and the longest unbroken run of them."""
    missing = [y for y in EXPECTED_YEARS if y not in present]
    longest = run = 0
    for year in EXPECTED_YEARS:
        run = run + 1 if year not in present else 0
        longest = max(longest, run)
    return missing, longest


### Three checks ported from the old Part B

In [ ]:
"""
CELL: Three more contradiction checks, ported from the old
Compendium_Data_Quality.ipynb's Part B. Its other three checks (totals vs.
parts, year-on-year jumps, a percent indicator holding counts) are not
ported separately - measure_chapter() below already finds exactly those
three, in one pass instead of a second one over the same table, under the
"total mismatch", "spike" and "units" issue kinds.

Adapted to take the table `measure_chapter()` has already prepared (a
"number" column already parsed, EN column names only - column names were
being looked up in English there too, so nothing is lost) rather than
re-parsing Value from scratch a second time.
"""


def check_implausible_values(have, chapter):
    """Negative counts, and percentages outside 0-100.

    An indicator is treated as a percentage if its name says so - '%',
    'percentage', 'proportion'.
    """
    if "Indicator" not in have.columns:
        return []

    name = have["Indicator"].astype(str).str.lower()
    is_percentage = name.str.contains(r"%|percentage|proportion", regex=True, na=False)

    negative = have[have["number"] < 0]
    out_of_range = have[is_percentage & ((have["number"] < 0) | (have["number"] > 100))]

    issues = []
    for row in negative.itertuples(index=False):
        issues.append({
            "chapter": chapter, "kind": "implausible value",
            "indicator": getattr(row, "Indicator", ""), "country": getattr(row, "Country", ""),
            "detail": f"negative value: {row.number:,.1f}",
            "severity": round(abs(row.number), 1),
        })
    for row in out_of_range.itertuples(index=False):
        issues.append({
            "chapter": chapter, "kind": "implausible value",
            "indicator": getattr(row, "Indicator", ""), "country": getattr(row, "Country", ""),
            "detail": f"percentage outside 0-100: {row.number:,.1f}",
            "severity": round(max(-row.number, row.number - 100), 1),
        })
    return issues


def check_percentages_sum(have, chapter):
    """Percentage breakdowns that do not add up to 100 within their group.

    Only applied where a single dimension is clearly the thing being broken
    down, so a set of shares that silently drops a category is visible.
    """
    if "Indicator" not in have.columns:
        return []

    name = have["Indicator"].astype(str).str.lower()
    work = have[name.str.contains(r"%|percentage|proportion", regex=True, na=False)]
    if work.empty:
        return []

    issues = []
    for dimension, total_label in TOTAL_LABELS.items():
        if dimension not in work.columns:
            continue
        part = work[~work[dimension].isin([total_label, "Age unknown"]) & work[dimension].notna()]
        if part.empty:
            continue
        keys = [c for c in ["Indicator", "Country", "Year", "Sex"]
                if c in part.columns and c != dimension]
        if not keys:
            continue
        sums = part.groupby(keys, dropna=False, observed=True)["number"].sum(min_count=1).dropna()
        bad = sums[(sums - 100).abs() > PERCENT_TOLERANCE]
        for key, adds_up_to in bad.items():
            record = dict(zip(keys, key if isinstance(key, tuple) else (key,)))
            issues.append({
                "chapter": chapter, "kind": "percentages do not sum to 100",
                "indicator": record.get("Indicator", ""), "country": record.get("Country", ""),
                "detail": f"{dimension} adds up to {adds_up_to:.1f}%",
                "severity": round(abs(adds_up_to - 100), 1),
            })
    return issues


def check_duplicate_rows(have, chapter):
    """The same dimensions and year appearing twice with DIFFERENT values.

    Identical duplicates are harmless noise; conflicting ones mean one of the
    two figures is wrong and there is no way to tell which downstream.
    """
    keys = [c for c in have.columns if c not in FIXED and c != "number"]
    if not keys:
        return []

    counts = have.groupby(keys, dropna=False, observed=True)["number"].nunique()
    conflicting = counts[counts > 1]
    if conflicting.empty:
        return []

    issues = []
    for key, distinct_values in conflicting.head(500).items():
        record = dict(zip(keys, key if isinstance(key, tuple) else (key,)))
        issues.append({
            "chapter": chapter, "kind": "conflicting duplicate rows",
            "indicator": record.get("Indicator", ""), "country": record.get("Country", ""),
            "detail": (f"{distinct_values} distinct values reported for the same row - "
                       f"{', '.join(f'{k}={v}' for k, v in record.items() if k not in ('Indicator', 'Country'))}"),
            "severity": int(distinct_values),
        })
    return issues


### measure_chapter()

In [ ]:
"""
CELL: measure_chapter() - every metric for one chapter's long file.
"""


def measure_chapter(path):
    """Returns (series, dimensions, issues) for one <Chapter>_EN.xlsx."""
    chapter = path.name[: -len("_EN.xlsx")]
    table = pd.read_excel(path, engine="openpyxl")
    if "Indicator" not in table.columns:
        logger.warning(f"{path.name}: no Indicator column, skipping")
        return [], [], []

    dimensions = [c for c in table.columns if c not in FIXED]
    table["number"] = table["Value"].map(to_number)
    have = table[table["number"].notna()].copy()
    have["Year"] = have["Year"].astype(int)
    have["is_headline"] = headline_mask(have, dimensions)

    logger.info(f"{chapter}: {len(table):,} rows, {len(have):,} with a value, "
                f"{len(dimensions)} dimension(s)")

    series, dimension_rows, issues = [], [], []

    # -------------------------------------------------- levels 1 and 2
    # Every indicator x country the chapter could report, not only the pairs
    # that reported something. A pair with nothing is a real finding - a
    # 0-point series - and leaving it out of the denominator would quietly
    # flatter every summary built on top.
    reported = {key: group for key, group
                in have.groupby(["Indicator", "Country"], observed=True)}
    all_indicators = sorted(table["Indicator"].dropna().unique())
    all_countries = sorted(table["Country"].dropna().unique())

    for indicator in all_indicators:
        for country in all_countries:
            group = reported.get((indicator, country))

            if group is None:
                years_any, years_head = set(), set()
            else:
                years_any = set(group["Year"]) & set(EXPECTED_YEARS)
                years_head = set(group.loc[group["is_headline"], "Year"]) & set(EXPECTED_YEARS)
            missing, longest = missing_years_and_longest_gap(years_any)

            series.append({
                "chapter": chapter, "indicator": indicator, "country": country,
                "points": len(years_any),
                "headline_points": len(years_head),
                "band": band_of(len(years_any)),
                "coverage": round(len(years_any) / len(EXPECTED_YEARS) * 100, 1),
                "first_year": min(years_any) if years_any else None,
                "last_year": max(years_any) if years_any else None,
                "longest_gap": longest, "missing_years": missing,
                "values": 0 if group is None else len(group),
            })

            if group is None:
                continue
            for dimension in dimensions:
                total = TOTAL_LABELS.get(dimension)
                values = group[dimension].dropna().astype(str).str.strip()
                if total is not None:
                    values = values[values != total]
                if values.empty:
                    continue
                dimension_rows.append({
                    "chapter": chapter, "indicator": indicator, "country": country,
                    "dimension": dimension, "categories": values.nunique(),
                })

    # ------------------------------------------------------- dubious spikes
    # One value per series per year first: the same dimensions and year can
    # appear twice in these files, and without this the comparison below would
    # pit two rows from the SAME year against each other.
    keys = ["Indicator", "Country"] + dimensions
    tidy = (have.groupby(keys + ["Year"], dropna=False, observed=True)["number"]
            .first().reset_index().sort_values(keys + ["Year"]))
    grouped = tidy.groupby(keys, dropna=False, observed=True)
    tidy["previous"] = grouped["number"].shift(1)
    tidy["previous_year"] = grouped["Year"].shift(1)
    candidates = tidy[tidy["previous"].notna() & (tidy["previous"] != 0)
                      & (tidy["Year"] > tidy["previous_year"])].copy()
    ratio = candidates["number"] / candidates["previous"]
    for _, row in candidates[(ratio > SPIKE_FACTOR) | (ratio < 1 / SPIKE_FACTOR)].iterrows():
        issues.append({
            "chapter": chapter, "kind": "spike",
            "indicator": row["Indicator"], "country": row["Country"],
            "detail": (f"{int(row['previous_year'])}: {row['previous']:,.0f}"
                       f"  ->  {int(row['Year'])}: {row['number']:,.0f}"),
            "severity": round(abs(row["number"] / row["previous"]), 1),
        })

    # ------------------------------- percentage indicators holding counts
    name = have["Indicator"].astype(str).str.lower()
    percentages = have[name.str.contains(r"%|percentage|proportion", regex=True, na=False)]
    for indicator, group in percentages.groupby("Indicator", observed=True):
        over = group["number"] > 100
        if over.any() and not over.all():
            issues.append({
                "chapter": chapter, "kind": "units",
                "indicator": indicator, "country": "(several)",
                "detail": (f"{int(over.sum()):,} of {len(group):,} rows exceed 100 "
                           f"(largest {group['number'].max():,.0f}) - percentages and "
                           f"counts mixed in one indicator"),
                "severity": round(float(group["number"].max()), 0),
            })

    # ------------------------------ totals that contradict their own parts
    for dimension, total in TOTAL_LABELS.items():
        if dimension not in have.columns:
            continue
        keys2 = [c for c in ["Indicator", "Country", "Year", "Sex"]
                 if c in have.columns and c != dimension]
        work = have[["number", dimension] + keys2]
        label = work[dimension].astype(str).str.strip()
        totals = work[label == total].groupby(keys2, observed=True)["number"].first()
        parts = (work[work[dimension].notna() & (label != total) & (label != "Age unknown")]
                 .groupby(keys2, observed=True)["number"].sum(min_count=1))
        shared = totals.index.intersection(parts.index)
        if not len(shared):
            continue
        compare = pd.DataFrame({"total": totals.loc[shared], "parts": parts.loc[shared]})
        compare = compare[compare["total"].abs() > 0]
        gap = (compare["parts"] - compare["total"]).abs() / compare["total"].abs()
        for key, off in gap[gap > TOTAL_TOLERANCE].items():
            record = dict(zip(keys2, key if isinstance(key, tuple) else (key,)))
            issues.append({
                "chapter": chapter, "kind": "total mismatch",
                "indicator": record.get("Indicator", ""),
                "country": record.get("Country", ""),
                "detail": (f"{dimension} {record.get('Year', '')}: reported "
                           f"{compare.loc[key, 'total']:,.0f} vs parts summing to "
                           f"{compare.loc[key, 'parts']:,.0f}"),
                "severity": round(float(off) * 100, 1),
            })

    # ---------------------------------- the three checks from the old Part B
    issues += check_implausible_values(have, chapter)
    issues += check_percentages_sum(have, chapter)
    issues += check_duplicate_rows(have, chapter)

    return series, dimension_rows, issues


### Run - Part 2

In [ ]:
"""
CELL: Measure every chapter's final file.
"""
files = sorted(p for p in LONG_FILES_PATH.glob("*_EN.xlsx")
               if not p.name.endswith("_EN_questionnaires.xlsx"))
print(f"Reading {len(files)} chapter file(s) from {LONG_FILES_PATH}\n")

all_series, all_dimensions, all_issues = [], [], []
for path in files:
    s, d, i = measure_chapter(path)
    all_series += s
    all_dimensions += d
    all_issues += i

SERIES = pd.DataFrame(all_series)
DIMENSIONS = pd.DataFrame(all_dimensions)
ISSUES = pd.DataFrame(all_issues)

if SERIES.empty:
    print("\nNo data. Run the pipeline notebooks first.")
else:
    print(f"\n{len(SERIES):,} series - {len(DIMENSIONS):,} disaggregation entries "
          f"- {len(ISSUES):,} flagged figures")

    print(f"\nHow many of the {len(EXPECTED_YEARS)} years each series actually has:")
    counts = SERIES["points"].value_counts().sort_index(ascending=False)
    for points, n in counts.items():
        bar = "#" * max(1, round(n / counts.max() * 40))
        note = "  <- complete" if points == len(EXPECTED_YEARS) else (
               "  <- nothing at all" if points == 0 else "")
        print(f"   {points:>2} points  {n:>5,}  {bar}{note}")

    print("\nBy band:")
    for name, low, high in POINT_BANDS:
        n = (SERIES["band"] == name).sum()
        span = f"{low}" if low == high else f"{low}-{high}"
        print(f"   {name:<9} ({span:>5} points)  {n:>5,}  {n/len(SERIES):>5.1%}")

    print("\nComplete series by chapter:")
    for chapter, group in SERIES.groupby("chapter"):
        full = (group["points"] == len(EXPECTED_YEARS)).sum()
        none = (group["points"] == 0).sum()
        print(f"   {chapter:<12} {full:>4,} complete - {none:>4,} empty - "
              f"{len(group):>4,} total")

    print(f"\nSeries reporting nothing since 2019: "
          f"{(SERIES['last_year'] < 2020).sum():,}")
    if len(ISSUES):
        print("\nFlagged figures by kind:")
        print(ISSUES["kind"].value_counts().to_string())


### The Excel report

In [ ]:
"""
CELL: Write the Excel report.
"""
if not SERIES.empty:
    with pd.ExcelWriter(GAPS_REPORT_PATH, engine="openpyxl") as writer:
        summary = pd.DataFrame(
            [{"metric": "series (every indicator x country)", "value": len(SERIES)}]
            + [{"metric": f"{name} ({low} points)" if low == high
                          else f"{name} ({low}-{high} points)",
                "value": int((SERIES["band"] == name).sum())}
               for name, low, high in reversed(POINT_BANDS)]
            + [{"metric": "series stale since 2019",
                "value": int((SERIES["last_year"] < 2020).sum())},
               {"metric": "flagged figures", "value": len(ISSUES)}])
        summary.to_excel(writer, sheet_name="summary", index=False)

        # One row per point count, so the shape of the distribution is
        # readable at a glance.
        distribution = (SERIES["points"].value_counts().reindex(
            range(len(EXPECTED_YEARS), -1, -1), fill_value=0)
            .rename_axis("points").reset_index(name="series"))
        distribution["share"] = (distribution["series"] / len(SERIES)).round(3)
        distribution.to_excel(writer, sheet_name="points distribution", index=False)

        (SERIES.pivot_table(index="country", columns="band", values="indicator",
                            aggfunc="count", fill_value=0)
         .reindex(columns=[b[0] for b in reversed(POINT_BANDS)], fill_value=0)
         .to_excel(writer, sheet_name="bands by country"))
        (SERIES.pivot_table(index="indicator", columns="band", values="country",
                            aggfunc="count", fill_value=0)
         .reindex(columns=[b[0] for b in reversed(POINT_BANDS)], fill_value=0)
         .to_excel(writer, sheet_name="bands by indicator"))

        out = SERIES.copy()
        out["missing_years"] = out["missing_years"].map(
            lambda years: ", ".join(str(y) for y in years))
        out.sort_values("points").to_excel(writer, sheet_name="series", index=False)

        if not DIMENSIONS.empty:
            DIMENSIONS.to_excel(writer, sheet_name="disaggregation", index=False)
        if not ISSUES.empty:
            (ISSUES.sort_values("severity", ascending=False)
             .to_excel(writer, sheet_name="flagged figures", index=False))

    print(f"Wrote {GAPS_REPORT_PATH.name}")
    print("  sheets: summary, points distribution, series, bands by country,")
    print("          bands by indicator, disaggregation, flagged figures")
